In [ ]:
import argparse
import re
import pandas as pd
from pathlib import Path
from collections import defaultdict
import json
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import matplotlib.dates as mdates
from datetime import datetime


In [ ]:
input_folder = Path("/home/kix/umu/dataset_art_datos/06_csv_csv")
input_file = input_folder

In [ ]:
# File pattern to extract date from filename
FILE_PATTERN = re.compile(r".*_(\d{8})_.*\.csv$")

In [ ]:
def get_files(inputs):
    """
    inputs could be:
    - a string with a file or directory
    - a list of strings
    - mixed Paths

    Returns a list of Paths to valid CSV files.
    """
    if isinstance(inputs, (str, Path)):
        inputs = [inputs]

    result = set()  # avoid duplicates

    for item in inputs:
        p = Path(item)

        # If it's a directory → search for CSV files that match your pattern
        if p.is_dir():
            for f in p.glob("*.csv"):
                if FILE_PATTERN.match(f.name):
                    result.add(f.resolve())

        # If it's a file → check if it matches
        elif p.is_file():
            if FILE_PATTERN.match(p.name):
                result.add(p.resolve())
            else:
                print(f"Skipped (pattern does not match): {p}")

        # If it does not exist
        else:
            print(f"Warning: {p} does not exists.")

    return sorted(result)

In [ ]:
files = get_files(input_file)
print(files)

In [ ]:
def extract_day(filename):
    """Extracts the day (YYYYMMDD) from the filename."""
    m = FILE_PATTERN.match(filename)
    if not m:
        return None
    return m.group(1)

In [ ]:
def read_file(filepath):
    """
    Reads a CSV file and returns a pandas DataFrame.
    """
    df = pd.read_csv(filepath, sep=';', header=None, dtype=str, engine='python')
    return df

In [ ]:
MODIFIED_FIELDS = [
    'timestamp',
    'ip_version', 'ip_ihl', 'ip_orig_tos', 'ip_tos',
    'ip_prec',
    'ip_dscp',
    'ip_enc', 'ip_len', 'ip_id', 'ip_orig_flags', 'ip_flag_MF', 'ip_flag_DF',
    'ip_flag_Error', 'ip_frag', 'ip_ttl', 'ip_proto', 'ip_chksum', 'ip_src',
    'ip_dst',
    'ip_options',
    'icmp_type', 'icmp_code', 'icmp_id', 'icmp_seq', 'icmp_chksum',
    'udp_sport', 'udp_dport', 'udp_len', 'udp_chksum',
    'tcp_sport', 'tcp_dport', 'tcp_seq', 'tcp_ack',
    'tcp_dataofs', 'tcp_reserved', 'tcp_orig_flags', 'tcp_CWR',
    'tcp_ACK_flag', 'tcp_PSH', 'tcp_RST', 'tcp_SYN', 'tcp_URG', 'tcp_FIN',
    'tcp_ECE', 'tcp_NS', 'tcp_other_flags', 'tcp_window',
    'tcp_chksum', 'tcp_urgptr', 'tcp_options', 'payload_hex',
    'key', 'IBR_info']

In [ ]:
def process_ip_proto(df, day, results_proto):
    """
    Processes the 'ip_proto' column in the DataFrame and counts occurrences of each protocol.
    Updates results_proto in place.
    """
    # Most used protocols
    column = MODIFIED_FIELDS.index('ip_proto')
    wk_df = pd.to_numeric(df[column], errors='coerce')

    counts = wk_df.value_counts()

    for proto, count in counts.items():
        proto = int(proto)
        results_proto[day][proto] += int(count)


In [ ]:
def process_ip_src(df, day, results_ip_src):
    """
    Processes the 'ip_src' column in the DataFrame and counts occurrences of each source IP.
    Updates results_ip_src in place.
    """
    # Different IP sources
    column = MODIFIED_FIELDS.index('ip_src')
    wk_df = df[column]

    counts = wk_df.value_counts()

    for ip_src, count in counts.items():
        results_ip_src[day][ip_src] += int(count)


In [ ]:
def process_ip_dst(df, day, results_ip_dst):
    """
    Processes the 'ip_dst' column in the DataFrame and counts occurrences of each destination IP.
    Updates results_ip_dst in place.
    """
    # Different IP destinations
    column = MODIFIED_FIELDS.index('ip_dst')
    wk_df = df[column]

    counts = wk_df.value_counts()

    for ip_dst, count in counts.items():
        results_ip_dst[day][ip_dst] += int(count)

In [ ]:
def process_ip_5min(df, day, results_ip_5min):
    """
    Processes traffic in 5-minute intervals.
    Stores the timestamp and the number of packets received in each 5-minute slot.
    Updates results_ip_5min in place.
    """
    wk_df = df[[MODIFIED_FIELDS.index('timestamp')]].copy()
    wk_df.columns = ['timestamp']
    wk_df['timestamp'] = pd.to_datetime(pd.to_numeric(wk_df['timestamp'], errors='coerce'), unit='s')

    # Create 5-minute blocks
    wk_df['time_bucket'] = wk_df['timestamp'].dt.floor('5min')

    # Count packets per 5-minute block
    for time_bucket, count in wk_df.groupby('time_bucket').size().items():
        time_key = time_bucket.strftime('%Y-%m-%d %H:%M:%S')
        results_ip_5min[day][time_key] = count

In [ ]:
def process_udp_dports(df, day, results_udp_ports):
    """
    Processes 'udp_dport' columns in the DataFrame and counts occurrences of each UDP port.
    Updates results_udp_ports in place.
    """
    # Filter the udp protocol
    wk_df = df.copy()
    tcp_col = MODIFIED_FIELDS.index('ip_proto')
    wk_df = wk_df[wk_df[tcp_col] == '17']

    column = MODIFIED_FIELDS.index('udp_dport')
    wk_df = pd.to_numeric(wk_df[column], errors='coerce')

    counts = wk_df.value_counts()

    for port, count in counts.items():
        port = int(port)
        results_udp_ports[day][port] += int(count)

In [ ]:
def process_tcp_dports(df, day, results_tcp_ports):
    """
    Processes 'tcp_dport' column in the DataFrame and counts occurrences of each TCP destination port.
    Updates results_tcp_ports in place.
    """
    # Filter the TCP protocol (ip_proto == 6)
    wk_df = df.copy()
    proto_col = MODIFIED_FIELDS.index('ip_proto')
    wk_df = wk_df[wk_df[proto_col] == '6']

    column = MODIFIED_FIELDS.index('tcp_dport')
    wk_df = pd.to_numeric(wk_df[column], errors='coerce')

    counts = wk_df.value_counts()

    for port, count in counts.items():
        port = int(port)
        results_tcp_ports[day][port] += int(count)

In [ ]:
def process_icmp_types(df, day, results_icmp_types):
    """
    Processes 'icmp_type' column in the DataFrame and counts occurrences of each ICMP type.
    Updates results_icmp_types in place.
    """
    # Filter the ICMP protocol (ip_proto == 1)
    wk_df = df.copy()
    proto_col = MODIFIED_FIELDS.index('ip_proto')
    wk_df = wk_df[wk_df[proto_col] == '1']

    column = MODIFIED_FIELDS.index('icmp_type')
    wk_df = pd.to_numeric(wk_df[column], errors='coerce')

    counts = wk_df.value_counts()

    for icmp_type, count in counts.items():
        icmp_type = int(icmp_type)
        results_icmp_types[day][icmp_type] += int(count)

In [ ]:
def process_icmp_types_and_codes(df, day, results_icmp_types_and_codes):
    """
    Processes 'icmp_type' and 'icmp_code' columns in the DataFrame and counts occurrences of each ICMP type and code combination.
    Updates results_icmp_types_and_codes in place.
    """
    type_column = MODIFIED_FIELDS.index('icmp_type')
    code_column = MODIFIED_FIELDS.index('icmp_code')

    # Filter the icmp protocol
    wk_df = df.copy()
    tcp_col = MODIFIED_FIELDS.index('ip_proto')
    wk_df = wk_df[wk_df[tcp_col] == '1']

    wk_df = wk_df[[type_column, code_column]].copy()
    wk_df.columns = ['icmp_type', 'icmp_code']
    wk_df['icmp_type'] = pd.to_numeric(wk_df['icmp_type'], errors='coerce')
    wk_df['icmp_code'] = pd.to_numeric(wk_df['icmp_code'], errors='coerce')

    # Remove rows with NaN values
    wk_df = wk_df.dropna()

    # Create a combined key of type and code
    wk_df['type_code'] = wk_df.apply(lambda row: (int(row['icmp_type']), int(row['icmp_code'])), axis=1)

    counts = wk_df['type_code'].value_counts()

    for (icmp_type, icmp_code), count in counts.items():
        results_icmp_types_and_codes[day][(icmp_type, icmp_code)] += int(count)

In [ ]:
def process_tcp_flags(df, day, results_tcp_flags):
    """
    Processes TCP flag columns in the DataFrame and counts occurrences of each flag combination.
    Interpreta correctamente valores True/False (bool reales o cadenas) y construye
    combinaciones de flags. Incluye tcp_other_flags cuando existe.
    Actualiza results_tcp_flags in place.
    """
    col_tcp_cwr = MODIFIED_FIELDS.index('tcp_CWR')
    col_tcp_ack = MODIFIED_FIELDS.index('tcp_ACK_flag')
    col_tcp_psh = MODIFIED_FIELDS.index('tcp_PSH')
    col_tcp_rst = MODIFIED_FIELDS.index('tcp_RST')
    col_tcp_syn = MODIFIED_FIELDS.index('tcp_SYN')
    col_tcp_urg = MODIFIED_FIELDS.index('tcp_URG')
    col_tcp_fin = MODIFIED_FIELDS.index('tcp_FIN')
    col_tcp_ece = MODIFIED_FIELDS.index('tcp_ECE')
    col_tcp_ns  = MODIFIED_FIELDS.index('tcp_NS')
    col_tcp_other = MODIFIED_FIELDS.index('tcp_other_flags')

    # Filter the tcp protocol
    wk_df = df.copy()
    tcp_col = MODIFIED_FIELDS.index('ip_proto')
    wk_df = wk_df[wk_df[tcp_col] == '6']
    # print(wk_df)

    wk_df = wk_df[[col_tcp_cwr, col_tcp_ack, col_tcp_psh, col_tcp_rst, col_tcp_syn, col_tcp_urg,
                col_tcp_fin, col_tcp_ece, col_tcp_ns, col_tcp_other]].copy()
    wk_df.columns = ['CWR', 'ACK', 'PSH', 'RST', 'SYN', 'URG', 'FIN', 'ECE', 'NS', 'OTHER']

    def to_bool_int(val):
        if isinstance(val, bool):
            return 1 if val else 0
        if pd.isna(val):
            return 0
        s = str(val).strip().lower()
        if s in ('1','true','t','yes','y'): return 1
        if s in ('0','false','f','no','n',''): return 0
        try:
            num = float(s)
            return 1 if num != 0 else 0
        except ValueError:
            return 0

    # Convert flag columns to 0/1 while preserving True/False semantics
    for col in ['CWR','ACK','PSH','RST','SYN','URG','FIN','ECE','NS']:
        wk_df[col] = wk_df[col].map(to_bool_int)

    def build_flag_combo(row):
        flags = []
        if row['SYN']: flags.append('SYN')
        if row['ACK']: flags.append('ACK')
        if row['FIN']: flags.append('FIN')
        if row['RST']: flags.append('RST')
        if row['PSH']: flags.append('PSH')
        if row['URG']: flags.append('URG')
        if row['ECE']: flags.append('ECE')
        if row['CWR']: flags.append('CWR')
        if row['NS']:  flags.append('NS')
        other = str(row['OTHER']).strip()
        if other and other not in ('0','', 'nan', 'None'):  # attach other flags descriptor
            flags.append(f'OTHER:{other}')
        return ','.join(flags) if flags else 'NONE'

    wk_df['flag_combination'] = wk_df.apply(build_flag_combo, axis=1)

    for combo, count in wk_df['flag_combination'].value_counts().items():
        results_tcp_flags[day][combo] += int(count)

In [ ]:
def counter_protocol(files):
    results_proto = defaultdict(lambda: defaultdict(int))
    results_ip_src = defaultdict(lambda: defaultdict(int))
    results_ip_dst = defaultdict(lambda: defaultdict(int))
    results_ip_5min = defaultdict(lambda: defaultdict(int))
    results_udp_dports = defaultdict(lambda: defaultdict(int))
    results_tcp_dports = defaultdict(lambda: defaultdict(int))
    results_icmp_types = defaultdict(lambda: defaultdict(int))
    results_icmp_types_and_codes = defaultdict(lambda: defaultdict(int))
    results_tcp_flags = defaultdict(lambda: defaultdict(int))

    for f in files:
        print(f"Processing: {f}")
        day = extract_day(f.name)
        if not day:
            print(f"Skipped (no match): {f}")
            continue

        try:
            df = read_file(f)

            # Count protocols in the file, results_proto is updated in place
            process_ip_proto(df, day, results_proto)

            # Count different IP sources, results_ip_src is updated in place
            process_ip_src(df, day, results_ip_src)

            # Count different IP destinations, results_ip_dst is updated in place
            process_ip_dst(df, day, results_ip_dst)

            # Count traffic every 5 minutes, results_ip_5min is updated in place
            process_ip_5min(df, day, results_ip_5min)

            # Count UDP ports, results_udp_ports is updated in place
            process_udp_dports(df, day, results_udp_dports)

            # Count TCP ports, results_tcp_ports is updated in place
            process_tcp_dports(df, day, results_tcp_dports)

            # Count ICMP types, results_icmp_types is updated in place
            process_icmp_types(df, day, results_icmp_types)

            # Count ICMP types and codes, results_icmp_types_and_codes is updated in place
            process_icmp_types_and_codes(df, day, results_icmp_types_and_codes)

            # Count TCP flags, results_tcp_flags is updated in place
            process_tcp_flags(df, day, results_tcp_flags)
        except Exception as e:
            print(f"Error processing {f}: {e}")

    # Convert defaultdict to regular dict for cleaner output
    dict_results = {
        'ip_proto': {day: dict(proto_counts) for day, proto_counts in results_proto.items()},
        'ip_src': {day: dict(ip_counts) for day, ip_counts in results_ip_src.items()},
        'ip_dst': {day: dict(ip_counts) for day, ip_counts in results_ip_dst.items()},
        'ip_5min': {day: dict(time_counts) for day, time_counts in results_ip_5min.items()},
        'udp_dports': {day: dict(port_counts) for day, port_counts in results_udp_dports.items()},
        'tcp_dports': {day: dict(port_counts) for day, port_counts in results_tcp_dports.items()},
        'icmp_types': {day: dict(type_counts) for day, type_counts in results_icmp_types.items()},
        'icmp_types_and_codes': {day: dict(type_code_counts) for day, type_code_counts in results_icmp_types_and_codes.items()},
        'tcp_flags': {day: dict(flag_counts) for day, flag_counts in results_tcp_flags.items()},
    }

    return dict_results


In [ ]:
def print_results(results):
    print("\n=== FINAL RESULTS ===")
    for day in sorted(results):
        print(f"{day}: {dict(results[day])}")

In [ ]:
def aggregate_results(results):
    """
    Aggregates all values across all days for both ip_proto and ip_src.
    Returns a dictionary with aggregated counts.
    """
    aggregated = {}

    for key, day_data in results.items():
        totals = defaultdict(int)

        for day, counts in day_data.items():
            for item, count in counts.items():
                totals[item] += count

        # Convert to regular dict and sort by count (descending)
        aggregated[key] = dict(sorted(totals.items(), key=lambda x: x[1], reverse=True))
    return aggregated

In [ ]:
results = counter_protocol(files)

In [ ]:

def main():
    parser = argparse.ArgumentParser(description="Count protocols per day in darknet CSV files")
    parser.add_argument("inputs", nargs="+", help="Input files or directories")
    args = parser.parse_args()

    files = get_files(args.inputs)

    if not files:
        print("No valid files found.")
        return

    results = counter_protocol(files)
    for key, res in results.items():
        print(f"\nResults for {key}:")

In [ ]:
# Just debug
"""
for key, res in results.items():
    for k2, r2 in res.items():
        print(f"Results for {key}, day {k2}: {r2}")
"""

In [ ]:
PROTO_NAMES = {
    1: "  1 - ICMP",
    4: "  4 - IP",
    6: "  6 - TCP",
    17: " 17 - UDP",
    41: " 41 - IPv6",
    47: " 47 - GRE",
    132: "132 - SCTP",
    138: "138 - MPLS",
}

PALETTE = ["#5c94bd", "#ff7f0e", "#49e4dcc0", "#dd5a5a", "#e7eb30", "#8c564b", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]

In [ ]:
# Aggregate results across all days
"""
aggregated = aggregate_results(results)

print("\n=== AGGREGATED RESULTS (All days combined) ===\n")
for key, totals in aggregated.items():
    print(f"\n{key.upper()}:")
    for item, count in totals.items():
        if key == 'ip_proto':
            proto_name = PROTO_NAMES.get(item, f"{item:3}")
            print(f"  {proto_name}: {count:,}")
        else:
            print(f"  {item}: {count:,}")
"""

In [ ]:
def save_results_to_json(results, output_file):
    """
    Save the results dictionary to a JSON file.
    """
    with open(output_file, "w") as f:
        json.dump({day: dict(results[day]) for day in results}, f, indent=2)

    print(f"Results saved to {output_file}")

In [ ]:
def save_results_to_json(results, output_file):
    """
    Save the results dictionary to a JSON file.
    Handles tuple keys by converting them to strings.
    """
    serializable_results = {}

    for day, counts in results.items():
        day_data = {}
        for key, value in counts.items():
            # Convert tuples to strings for JSON serialization
            if isinstance(key, tuple):
                day_data[str(key)] = value
            else:
                day_data[key] = value
        serializable_results[day] = day_data

    with open(output_file, "w") as f:
        json.dump(serializable_results, f, indent=2)

    print(f"Results saved to {output_file}")

In [ ]:
save_results_to_json(results['ip_proto'], "protocol_counts.json")
save_results_to_json(results['ip_src'], "ip_src_counts.json")
save_results_to_json(results['ip_dst'], "ip_dst_counts.json")
save_results_to_json(results['ip_5min'], "ip_5min_counts.json")
save_results_to_json(results['udp_dports'], "udp_dport_counts.json")
save_results_to_json(results['tcp_dports'], "tcp_dport_counts.json")
save_results_to_json(results['tcp_flags'], "tcp_flag_counts.json")
save_results_to_json(results['icmp_types'], "icmp_type_counts.json")
save_results_to_json(results['icmp_types_and_codes'], "icmp_types_and_codes_counts.json")

In [ ]:
def plot_protocols_stacked(json_file):
    fontsize = 20
    # Load data
    with open(json_file, "r") as f:
        data = json.load(f)

    # Convert protocol keys to int
    new_data = {}
    for day, proto_counts in data.items():
        new_data[day] = {int(proto): count for proto, count in proto_counts.items()}

    # Create DataFrame
    df = pd.DataFrame(new_data).T  # days as rows, protocols as columns
    df.index.name = "day"
    df.fillna(0, inplace=True)

    # Sort columns by protocol number
    df = df.reindex(sorted(df.columns), axis=1)

    # Rename columns according to PROTO_NAMES (if no name, use number)
    df.rename(columns=lambda x: PROTO_NAMES.get(x, str(x)), inplace=True)

    # Format date index
    # df.index = pd.to_datetime(df.index, format="%Y%m%d").strftime("%d-%b").str.lower()
    # idx = pd.to_datetime(df.index, format="%Y%m%d").strftime("%d-%b").str.lower()
    # df.index = (
    #     idx.str.replace("-oct", "-Oct", regex=False)
    #        .str.replace("-nov", "-Nov", regex=False)
    #)
    df.index = pd.to_datetime(df.index, format="%Y%m%d").strftime("%d-%m-%Y")

    # Draw stacked bar chart
    ax = df.plot(
        kind="bar",
        stacked=True,
        figsize=(14,8),
        color=PALETTE[:len(df.columns)],
        width=0.85,
    )


    # Convert numbers from matplotlib to datetime
    print(df.index)
    xnum = mdates.date2num(pd.to_datetime(df.index, format="%d-%m-%Y").to_list())
    print(xnum)
    dates = [mdates.num2date(d) for d in xnum]

    # Create a boolean mask for alternating days
    alternate = True
    days = []

    # Add alternate days to the list
    for d in dates:
        if alternate:
            d.day
            days.append(d)
        alternate = not alternate

    # Filter only odd days
    # days = [d for d in dates if d.day % 2 != 1]

    # Use only those days as ticks
    ax.set_xticks(days)





    # ax.title.set_size(16)
    ax.xaxis.label.set_size(fontsize + 2)
    ax.yaxis.label.set_size(fontsize + 2)
    ax.tick_params(axis='x', labelsize=fontsize + 2)
    ax.tick_params(axis='y', labelsize=fontsize + 2)

    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: (y/1000000)))

    ax.set_xlabel("Day", fontsize=fontsize + 2)
    ax.set_ylabel("Packets (millions)", fontsize=fontsize + 2)
    # ax.set_title("Packets by Protocol per Day")
    plt.xticks(rotation=45, ha="right")
    plt.legend(title="Protocol",
               fontsize=fontsize,
               title_fontsize=fontsize,
               loc='lower center',
               labelspacing=0.3,
               handletextpad=0.5,
               borderpad=0.1,
               bbox_to_anchor=(0.5, 0.98),
               ncols=4)

    plt.tight_layout()
    # Save the figure
    plt.savefig("protocols_stacked.pdf", dpi=300, bbox_inches='tight')

    plt.show()

In [ ]:
def plot_protocols_stacked(json_file):
    fontsize = 20
    with open(json_file, "r") as f:
        data = json.load(f)

    # Convert protocol keys to int
    new_data = {day: {int(proto): count for proto, count in proto_counts.items()}
                for day, proto_counts in data.items()}

    df = pd.DataFrame(new_data).T
    if df.empty:
        print("No data to plot.")
        return

    df.index.name = "day"
    df.fillna(0, inplace=True)

    # Sort columns by protocol number
    df = df.reindex(sorted(df.columns), axis=1)

    # Rename columns
    df.rename(columns=lambda x: PROTO_NAMES.get(x, str(x)), inplace=True)

    # Date format DD-MM-YYYY
    dt_index = pd.to_datetime(df.index, format="%Y%m%d")
    df.index = dt_index.strftime("%d-%m-%Y")

    # Plot
    ax = df.plot(
        kind="bar",
        stacked=True,
        figsize=(14, 8),
        color=PALETTE[:len(df.columns)],
        width=0.85,
    )

    # Show only every 2 days (optional). Remove this block if you want all days.
    tick_positions = list(range(len(df.index)))
    visible_positions = tick_positions[::2]  # every 2
    ax.set_xticks(visible_positions)
    ax.set_xticklabels([df.index[i] for i in visible_positions], rotation=45, ha="right")

    # Axes and formatting
    ax.set_xlabel("Day", fontsize=fontsize + 2)
    ax.set_ylabel("Packets (millions)", fontsize=fontsize + 2)
    ax.tick_params(axis='x', labelsize=fontsize + 2)
    ax.tick_params(axis='y', labelsize=fontsize + 2)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: (y / 1_000_000)))

    # Legend outside at the top
    handles, labels = ax.get_legend_handles_labels()
    # ax.legend(handles, labels,
    #           title="Protocol",
    #           fontsize=fontsize,
    #           title_fontsize=fontsize,
    #           ncols=4,
    #           loc="upper center",
    #           bbox_to_anchor=(0.5, 1.18),
    #           labelspacing=0.3,
    #           handletextpad=0.5,
    #           borderpad=0.2,
    #           frameon=False)
    ax.legend(title="Protocol",
               fontsize=fontsize,
               title_fontsize=fontsize,
               loc='lower center',
               labelspacing=0.3,
               handletextpad=0.5,
               borderpad=0.1,
               bbox_to_anchor=(0.5, 0.98),
               ncols=4)

    plt.subplots_adjust(top=0.30)  # space for legend
    plt.tight_layout()

    plt.savefig("protocols_stacked.pdf", dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
plot_protocols_stacked("protocol_counts.json")

In [ ]:
def plot_5min_traffic(json_file):
    # Load data
    with open(json_file, "r") as f:
        data = json.load(f)

    for day, time_counts in data.items():
        # Convert to DataFrame
        df = pd.DataFrame(list(time_counts.items()), columns=["time", "packet_count"])
        df['time'] = pd.to_datetime(df['time'], format="%Y-%m-%d %H:%M:%S")

        # Draw line chart
        plt.figure(figsize=(14, 7))
        plt.plot(df['time'], df['packet_count'], marker='o', linestyle='-')
        plt.xlabel("Time", fontsize=16)
        plt.ylabel("Number of Packets", fontsize=16)
        plt.title(f"Traffic in 5-Minute Intervals for {day}", fontsize=18)
        plt.xticks(rotation=45)
        plt.grid(True)
        plt.tight_layout()
        plt.show()

In [ ]:
# This function read the 5 minutes intervals and draw a line plot of packets per 5 minutes interval, all days in one plot
# using only onle line for all days
def plot_5min_traffic(json_file):
    # Load data
    with open(json_file, "r") as f:
        data = json.load(f)

    # Create DataFrame
    all_data = []
    for day, time_counts in data.items():
        for time_str, count in time_counts.items():
            # time_str already contains full date and time (YYYY-MM-DD HH:MM:SS)
            timestamp = datetime.strptime(time_str, "%Y-%m-%d %H:%M:%S")
            all_data.append((timestamp, count))

    df = pd.DataFrame(all_data, columns=["timestamp", "count"])
    df.set_index("timestamp", inplace=True)
    df.sort_index(inplace=True)

    # Draw line chart
    plt.figure(figsize=(14,7))
    plt.plot(df.index, df['count'], marker='o', linestyle='-')

    plt.xlabel("Time", fontsize=16)
    plt.ylabel("Packets", fontsize=16)
    plt.title("Packets per 5-Minute Interval", fontsize=18)
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot 5-minute traffic using the correct JSON (ip_5min_counts.json)
plot_5min_traffic("ip_5min_counts.json")

In [ ]:
# Create a latex table with the top 25 TCP ports, with rank, port, count, and percentage of total
def create_tcp_ports_latex_table(json_file, output_file):
    # Load data
    with open(json_file, "r") as f:
        data = json.load(f)

    # Aggregate counts across all days
    total_counts = defaultdict(int)
    for day, port_counts in data.items():
        for port, count in port_counts.items():
            total_counts[int(port)] += int(count)

    # Sort ports by count (descending) and take top 25
    sorted_ports = sorted(total_counts.items(), key=lambda x: x[1], reverse=True)[:25]
    total_packets = sum(total_counts.values())

    # Create LaTeX table
    with open(output_file, "w") as f:
        f.write("\\begin{tabular}{|c|c|c|c|}\n")
        f.write("\\hline\n")
        f.write("Rank & TCP Port & Count & Percentage \\\\\n")
        f.write("\\hline\n")

        for rank, (port, count) in enumerate(sorted_ports, start=1):
            percentage = (count / total_packets) * 100
            f.write(f"{rank} & {port} & {count} & {percentage:.2f}\\% \\\\\n")

        f.write("\\hline\n")
        f.write("\\end{tabular}\n")

    print(f"LaTeX table saved to {output_file}")

In [ ]:
create_tcp_ports_latex_table("tcp_dport_counts.json", "top_25_tcp_ports.tex")

In [ ]:
# Create a latex table with the top 25 UDP ports, with rank, port, count, and percentage of total
def create_udp_ports_latex_table(json_file, output_file):
    # Load data
    with open(json_file, "r") as f:
        data = json.load(f)

    # Aggregate counts across all days
    total_counts = defaultdict(int)
    for day, port_counts in data.items():
        for port, count in port_counts.items():
            total_counts[int(port)] += int(count)

    # Sort ports by count (descending) and take top 25
    sorted_ports = sorted(total_counts.items(), key=lambda x: x[1], reverse=True)[:25]
    total_packets = sum(total_counts.values())

    # Create LaTeX table
    with open(output_file, "w") as f:
        f.write("\\begin{tabular}{|c|c|c|c|}\n")
        f.write("\\hline\n")
        f.write("Rank & UDP Port & Count & Percentage \\\\\n")
        f.write("\\hline\n")

        for rank, (port, count) in enumerate(sorted_ports, start=1):
            percentage = (count / total_packets) * 100
            f.write(f"{rank} & {port} & {count} & {percentage:.2f}\\% \\\\\n")

        f.write("\\hline\n")
        f.write("\\end{tabular}\n")

    print(f"LaTeX table saved to {output_file}")

In [ ]:
create_udp_ports_latex_table("udp_dport_counts.json", "top_25_udp_ports.tex")

In [ ]:
# This function creates a LaTeX table for TCP flag combinations
# One column per flag: A (ACK), C (CWR), E (ECE), F (FIN), N (NS), P (PSH), R (RST), S (SYN), U (URG)
# Shows letter when flag is active, space when inactive (ordered alphabetically)
# Plus columns for Rank, Count, and Percentage

def create_tcp_flags_latex_table(json_file, output_file, top_n=25):
    no_selected_flag_character = ' '
    with open(json_file, "r") as f:
        data = json.load(f)

    # Aggregate counts across all days
    total_counts = defaultdict(int)
    for day, combo_counts in data.items():
        for combo, count in combo_counts.items():
            total_counts[combo] += int(count)

    # Sort combos by count desc and take top N
    sorted_combos = sorted(total_counts.items(), key=lambda x: x[1], reverse=True)[:top_n]
    total_packets = sum(total_counts.values()) if total_counts else 0

    # Parse combination string into flag dict
    def parse_flags(combo_str):
        # Returns dict with True/False for each flag
        flags = {'A': False, 'C': False, 'E': False, 'F': False, 'N': False,
                 'P': False, 'R': False, 'S': False, 'U': False}
        if combo_str == 'NONE':
            return flags

        parts = combo_str.split(',')
        for p in parts:
            if p == 'ACK': flags['A'] = True
            elif p == 'CWR': flags['C'] = True
            elif p == 'ECE': flags['E'] = True
            elif p == 'FIN': flags['F'] = True
            elif p == 'NS': flags['N'] = True
            elif p == 'PSH': flags['P'] = True
            elif p == 'RST': flags['R'] = True
            elif p == 'SYN': flags['S'] = True
            elif p == 'URG': flags['U'] = True
        return flags

    with open(output_file, 'w') as f:
        # Header with one column per flag (alphabetically ordered)
        f.write("\\begin{tabular}{|c|c|c|c|c|c|c|c|c|c|r|c|}\n")
        f.write("\\hline\n")
        f.write("Rank & A & C & E & F & N & P & R & S & U & Count & \\% \\\\\n")
        f.write("\\hline\n")

        for rank, (combo, count) in enumerate(sorted_combos, start=1):
            pct = (count / total_packets * 100) if total_packets else 0
            flags = parse_flags(combo)

            # Build row: rank, then each flag column (letter or space), count, percentage
            # Flags in alphabetical order: A, C, E, F, N, P, R, S, U
            row = f"{rank}"
            for flag_letter in ['A', 'C', 'E', 'F', 'N', 'P', 'R', 'S', 'U']:
                cell = flag_letter if flags[flag_letter] else no_selected_flag_character
                row += f" & {cell}"
            row += f" & {count} & {pct:.2f}\\%"
            f.write(row + " \\\\\n")

        f.write("\\hline\n")
        f.write("\\end{tabular}\n")

    print(f"LaTeX TCP flags table saved to {output_file}")



In [ ]:
# Generate the table
create_tcp_flags_latex_table("tcp_flag_counts.json", "tcp_flag_combinations.tex")

In [ ]:
# This function creates a LaTeX table for ICMP types and codes
def create_icmp_types_codes_latex_table(json_file, output_file, top_n=25):
    with open(json_file, "r") as f:
        data = json.load(f)

    # Aggregate counts across all days
    total_counts = defaultdict(int)
    for day, type_code_counts in data.items():
        for type_code_str, count in type_code_counts.items():
            type_code = eval(type_code_str)  # Convert string back to tuple
            total_counts[type_code] += int(count)

    # Sort by count desc and take top N
    sorted_type_codes = sorted(total_counts.items(), key=lambda x: x[1], reverse=True)[:top_n]
    total_packets = sum(total_counts.values()) if total_counts else 0

    with open(output_file, 'w') as f:
        f.write("\\begin{tabular}{|c|c|c|r|c|}\n")
        f.write("\\hline\n")
        f.write("Rank & ICMP Type & ICMP Code & Count & \\% \\\\\n")
        f.write("\\hline\n")

        for rank, ((icmp_type, icmp_code), count) in enumerate(sorted_type_codes, start=1):
            pct = (count / total_packets * 100) if total_packets else 0
            f.write(f"{rank} & {icmp_type} & {icmp_code} & {count} & {pct:.2f}\\% \\\\\n")

        f.write("\\hline\n")
        f.write("\\end{tabular}\n")

    print(f"LaTeX ICMP types and codes table saved to {output_file}")

In [ ]:
create_icmp_types_codes_latex_table("icmp_types_and_codes_counts.json", "icmp_types_codes.tex")

In [ ]:
# This LaTeX table shows the ip protocol counts aggregated across all days and include the percentage of total packets for each protocol.
def create_ip_proto_latex_table(json_file, output_file):
    with open(json_file, "r") as f:
        data = json.load(f)

    # Aggregate counts across all days
    total_counts = defaultdict(int)
    for day, proto_counts in data.items():
        for proto, count in proto_counts.items():
            total_counts[int(proto)] += int(count)

    # Sort by protocol number
    sorted_protos = sorted(total_counts.items(), key=lambda x: x[0])
    total_packets = sum(total_counts.values()) if total_counts else 0

    with open(output_file, 'w') as f:
        f.write("\\begin{tabular}{|l|r|c|}\n")
        f.write("\\hline\n")
        f.write("Protocol & Count & \\% \\\\\n")
        f.write("\\hline\n")

        for proto, count in sorted_protos:
            pct = (count / total_packets * 100) if total_packets else 0
            proto_name = PROTO_NAMES.get(proto, str(proto))
            f.write(f"{proto_name} ({proto}) & {count} & {pct:.2f}\\% \\\\\n")

        f.write("\\hline\n")
        f.write("\\end{tabular}\n")

    print(f"LaTeX IP protocol table saved to {output_file}")

In [ ]:
create_ip_proto_latex_table("protocol_counts.json", "ip_protocols.tex")

In [ ]:
# This function reads the ip_dst_counts.json and aggregates the counts across all days,
# then creates a plot of the 256 destinations as a bar chart.
def plot_top_ip_dst(json_file):
    with open(json_file, "r") as f:
        data = json.load(f)

    # Remove the "155.54.96." string from the keys if present
    cleaned_data = {}
    for day, ip_counts in data.items():
        cleaned_ip_counts = {}
        for ip_dst, count in ip_counts.items():
            if ip_dst.startswith("155.54.96."):
                cleaned_ip = ip_dst[len("155.54.96."):]
            else:
                cleaned_ip = ip_dst
            cleaned_ip_counts[cleaned_ip] = count
        cleaned_data[day] = cleaned_ip_counts

    # Aggregate counts across all days
    total_counts = defaultdict(int)
    for day, ip_counts in cleaned_data.items():
        for ip_dst, count in ip_counts.items():
            total_counts[ip_dst] += int(count)

    # Sort the dictionary by the key (is integer form of the last octet)
    total_counts = dict(sorted(total_counts.items(), key=lambda x: int(x[0])))

    print(total_counts)

    # Sort by count desc and take top 256
    # sorted_ips = sorted(total_counts.items(), key=lambda x: x[1], reverse=True)[:256]

    # Create DataFrame for plotting
    df = pd.DataFrame(total_counts.items(), columns=["ip_dst", "count"])
    df.set_index("ip_dst", inplace=True)

    # Print the median and mean of the counts
    print(f"Median of counts: {df['count'].median()}")
    print(f"Mean of counts: {df['count'].mean()}")
    print(f"Minimum of counts: {df['count'].min()}")
    print(f"Maximum of counts: {df['count'].max()}")

    # Plot bar chart
    plt.figure(figsize=(16, 8))
    df.plot(kind="bar", legend=False)

    # Show in the X-axis the xticks every 16 ticks
    plt.xticks(ticks=range(0, len(df.index), 16), labels=df.index[::16], rotation=90)

    plt.xlabel("Destination IP", fontsize=14)
    plt.ylabel("Packet Count", fontsize=14)
    plt.title("Top 256 Destination IPs by Packet Count", fontsize=16)
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()
    print(f"Plotted top 256 destination IPs.")

In [ ]:
plot_top_ip_dst("ip_dst_counts.json")

In [ ]:
# Top 25 IP source IP addresses as a LaTeX table
def create_top_ip_src_latex_table(json_file, output_file):
    with open(json_file, "r") as f:
        data = json.load(f)

    # Aggregate counts across all days
    total_counts = defaultdict(int)
    for day, ip_counts in data.items():
        for ip_src, count in ip_counts.items():
            total_counts[ip_src] += int(count)

    # Sort by count desc and take top 25
    sorted_ips = sorted(total_counts.items(), key=lambda x: x[1], reverse=True)[:25]
    total_packets = sum(total_counts.values()) if total_counts else 0

    with open(output_file, 'w') as f:
        f.write("\\begin{tabular}{|c|l|r|c|}\n")
        f.write("\\hline\n")
        f.write("Rank & Source IP & Count & \\% \\\\\n")
        f.write("\\hline\n")

        for rank, (ip_src, count) in enumerate(sorted_ips, start=1):
            pct = (count / total_packets * 100) if total_packets else 0
            f.write(f"{rank} & {ip_src} & {count} & {pct:.2f}\\% \\\\\n")

        f.write("\\hline\n")
        f.write("\\end{tabular}\n")

    print(f"LaTeX top source IPs table saved to {output_file}")


In [ ]:
create_top_ip_src_latex_table("ip_src_counts.json", "top_25_ip_src.tex")

In [ ]:
df